In [1]:
import chephy_model as cpm
import gnn_utils
import torch
import pandas as pd
import os
import re
from tqdm import tqdm

/home/dsi/chenbis/anaconda3/envs/chephy2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using cache found in /home/dsi/chenbis/.cache/torch/hub/facebookresearch_esm_main


In [2]:
atchley_path = "/home/dsi/chenbis/repos/sol_lab/files/atchley.csv"
atchley_df = pd.read_csv(atchley_path)
atchley_dict = atchley_df.set_index("amino.acid").to_dict(orient="index")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

patient_metadata_path = "../data_600_ab.csv"


In [ ]:
distance_threshold = 0.08
use_mst = True
enrich = True
cdr3_header = "aaSeqCDR3"

graph_path = "/dsi/efroni-lab/sbm/chen2/graphs_600_common"  
sub_folder = f"{graph_path}/{distance_threshold}_{use_mst}_{enrich}"
os.makedirs(sub_folder, exist_ok=True)

In [4]:
def prepare_data(data, cdr3_header):

    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    return data_filtered

In [5]:
from scipy.spatial.distance import squareform 

def get_graph_data_full_nodes(
    data, cdr3_header, label, distance_threshold=0.2, use_mst=True, enrich=False
):
    data = prepare_data(data, cdr3_header)

    # unique full sequences = nodes
    full_nodes = sorted(set(data[cdr3_header].dropna()))
    full_idx = {s: i for i, s in enumerate(full_nodes)}
    data["tcr_index"] = data[cdr3_header].map(full_idx)

    # build a mapping full_seq -> truncated_seq for distance computation
    tmp = pd.DataFrame({cdr3_header: full_nodes})
    tmp = cpm.truncate_sequences(tmp, cdr3_header, 4, 4)   # adds cdr3_truncated
    trunc_list = tmp["cdr3_truncated"].tolist()

    # compute distances between *truncated* strings, aligned to full_nodes order
    # easiest: feed the set, then reindex back (because your cpm returns its own order)
    trunc_set = set(trunc_list)
    chephy_matrix, trunc_order = cpm.find_che_phy_dist(trunc_set)
    trunc_idx = {s: i for i, s in enumerate(trunc_order)}

    # build full square distance matrix between nodes (full_nodes order)
    D_trunc = squareform(chephy_matrix)
    import numpy as np
    D = np.zeros((len(full_nodes), len(full_nodes)), dtype=float)
    for i, ti in enumerate(trunc_list):
        for j, tj in enumerate(trunc_list):
            D[i, j] = D_trunc[trunc_idx[ti], trunc_idx[tj]]

    # node weights (sum readFraction per full seq)
    node_weights = (
        data.dropna(subset=["tcr_index"])
            .groupby("tcr_index")["readFraction"].sum()
            .reindex(range(len(full_nodes)))
            .fillna(0.0)
            .to_numpy()
            .tolist()
    )

    graph = gnn_utils.build_graph(
        D,
        full_nodes,  # full sequences as node feature
        label=label,
        node_weights=node_weights,
        distance_threshold=distance_threshold,
        use_mst=use_mst,
        enrich=enrich
    )
    return graph, data


In [6]:
def patient_row_to_long_df(row, id_col, normalize=True):
    patient_id = str(row[id_col])

    # all sequence columns = everything except the id column
    seq_cols = [c for c in row.index if c != id_col]

    # coerce to numeric, ignore non-numeric junk
    vals = pd.to_numeric(row[seq_cols], errors="coerce").fillna(0.0)

    # keep sequences that are present (>0)
    present_mask = vals > 0
    present_seqs = vals.index[present_mask].tolist()
    present_vals = vals[present_mask].to_numpy(dtype=float)

    if len(present_seqs) == 0:
        # return empty df (caller can skip)
        return patient_id, pd.DataFrame(columns=[cdr3_header, "readFraction"])

    if normalize:
        s = float(present_vals.sum())
        if s > 0:
            present_vals = present_vals / s

    long_df = pd.DataFrame({
        cdr3_header: present_seqs,
        "readFraction": present_vals
    })
    return patient_id, long_df

In [7]:
def patient_label_from_id(patient_id: str) -> int:
    # Healthy ends with H, Sick ends with OC (based on your description)
    # Your file looks like "..._H" or "..._OC"
    if re.search(r'(^|_)OC$', patient_id):
        return 1
    if re.search(r'(^|_)H$', patient_id):
        return 0
    # fallback: if it contains OC anywhere at the end segment
    return 1 if patient_id.endswith("OC") else 0

In [8]:
wide = pd.read_csv(patient_metadata_path)

# first column is patient id (your file has 'Unnamed: 0')
id_col = wide.columns[0]

for _, row in tqdm(wide.iterrows(), total=len(wide), desc="Building patient graphs"):
    patient_id, long_df = patient_row_to_long_df(row, id_col=id_col, normalize=True)
    label = patient_label_from_id(patient_id)

    # skip patients with no sequences present after filtering
    if long_df.empty:
        continue

    graph, processed_long = get_graph_data_full_nodes(
        long_df,
        cdr3_header=cdr3_header,
        label=label,
        distance_threshold=distance_threshold,
        use_mst=use_mst,
        enrich=enrich
    )

    # save: (graph, processed_long, patient_id) so you can trace it later
    save_path = os.path.join(sub_folder, f"{patient_id}.pt")
    torch.save(
        {"patient_id": patient_id, "label": label, "graph": graph, "data": processed_long},
        save_path
    )

Building patient graphs:   0%|          | 0/83 [00:00<?, ?it/s]

Building patient graphs: 100%|██████████| 83/83 [00:08<00:00,  9.71it/s]


In [9]:
graph

Data(x=[102, 320], edge_index=[2, 386], y=[1], edge_weight=[386], node_weights=[102], node_seqs=[102])